In [ ]:
import sqlite3
import pandas as pd

pd.set_option('display.max_colwidth', None)

conn = sqlite3.connect("../data/election.db")
questions = pd.read_sql('SELECT id, axis, text FROM "QuizQuestion" ORDER BY id', conn)

PARTY = 'yisrael-beitenu'
platform = open(f'../data/platforms/{PARTY}.txt', encoding='utf-8').read()

In [ ]:
coverage = {
    'q03': ( 1, 'partial', 'הגדלת תקציב הביטחון מ-70 מיליארד ש"ח ל-95 מיליארד ש"ח בשנה'),  # לא אומר "על חשבון תקציבים אזרחיים"
    'q04': ( 0, 'unclear', 'הסכסוך של מדינת ישראל עם שכנותיה אינו סכסוך טריטוריאלי'),
    'q05': ( 1, 'partial', 'סגירת המעברים וניתוק אספקת המים, החשמל והדלק'),          # לא אומר "גם במחיר בידוד"
    'q07': (-1, 'partial', 'הפתרון לאתגרי הדיור בישראל הוא בעיקרו הגדלת ההיצע'),       # ← שינוי: היצע ותחרות במקום התערבות במחירים
    'q08': ( 1, 'partial', 'הפחתת רגולציה, צמצום משרדי הממשלה והמגזר הציבורי'),       # ← שינוי: היה "לא מצוין"
    'q10': ( 1, 'sure',    'השלמה להכנסה בגובה שכר מינימום לכלל הגמלאים בישראל'),      # ← שינוי: השורה שהייתה אצלך ב-q09
    'q11': ( 1, 'sure',    'העברת כלל הסמכויות בנושא פתיחת עסקים והפעלת תחבורה ציבורית בשבת לידי הרשויות המקומיות'),
    'q12': ( 1, 'sure',    'חוק גיוס לכולם'),
    'q13': ( 1, 'sure',    'מתן אפשרות לנישואין ולגירושין אזרחיים'),
    'q14': (-1, 'sure',    'מוסדות חינוך שלא ילמדו את תוכנית הליבה במלואה לא יהיו זכאים לתקצוב ממשלתי'),
    'q15': (-1, 'sure',    'יישום הרפורמה בתחום הכשרות'),                           # דורש ידע רקע: הרפורמה פותחת את הכשרות לתחרות
    'q19': ( 1, 'sure',    'ותבסס את עקרונות הדמוקרטיה באמצעות כינון חוקה'),
    'q20': ( 1, 'sure',    'הקמת ועדת חקירה ממלכתית לשבעה באוקטובר'),
    'q21': ( 0, 'unclear', 'המזדהה עם המדינה ותורם לה, זכאי למלוא הכבוד והזכויות'),  # ← שינוי: שוויון מותנה בנאמנות
    'q24': ( 1, 'sure',    'ענישה מרתיעה בגין הסתה נגד המדינה, השתתפות במפגשים עם אויבי ישראל'),  # ← שינוי: שורה 143, לא 113
}

In [ ]:
rows = []
for qid in questions['id']:
    if qid in coverage:
        stance, confidence, quote = coverage[qid]
    else:
        stance, confidence, quote = None, 'not_covered', None
    rows.append({'party': PARTY, 'question_id': qid,
                 'stance': stance, 'confidence': confidence, 'quote': quote})

result = questions.merge(pd.DataFrame(rows), left_on='id', right_on='question_id').drop(columns='id')

# בדיקה 1: כל ציטוט חייב להופיע במצע מילה במילה
missing = [q for q in result['quote'].dropna() if q not in platform]
print('ציטוטים שלא נמצאו במצע:', missing or 'אין')

# בדיקה 2: כל מפתח במילון הוא שאלה קיימת
print('מזהים לא מוכרים:', set(coverage) - set(questions['id']) or 'אין')

result[['question_id', 'axis', 'stance', 'confidence', 'quote']]

In [ ]:
result.to_csv(f'platform_coverage_{PARTY}.csv', index=False)

print(result['confidence'].value_counts())
print()
print(result.groupby('axis')['confidence'].apply(lambda s: (s != 'not_covered').sum()))

In [ ]:
import pandas as pd

# 1. פיצול לשורות
raw_lines = platform.split('\n')

# 2. ניקוי וסינון:
# - להוריד שורות ריקות (line.strip() != "")
# - להוריד שורות שמתחילות ב-# (לא כותרות URL/מפלגה)
# - להוריד כותרות של מילה אחת (יותר ממילה אחת)
# - להוריד שורות קצרות מאוד ("דלג לתוכן" הוא באורך 9 תווים, לכן נסנן לפי אורך > 10 או נסיר אותו מפורשות)
cleaned_lines = [
    line.strip()
    for line in raw_lines
    if line.strip()
    and not line.strip().startswith('#')
    and len(line.strip().split()) > 1      # מסנן כותרות של מילה אחת
    and len(line.strip()) > 10             # מסנן שורות קצרות מאוד (כגון "דלג לתוכן")
]

# המרה ל-Series לבדיקות
segments = pd.Series(cleaned_lines)

# 3. בדיקות:
print(f"כמות הקטעים שנשארו: {len(segments)}")
print("\nהתפלגות אורכי הקטעים (בתווים):")
print(segments.str.len().describe())


In [ ]:
# בדיקה: כל ציטוט מהתיוג הידני חייב להופיע בתוך קטע שלם
missing = []
for qid, (stance, confidence, quote) in coverage.items():
    found = any(quote in s for s in segments)
    if not found:
        missing.append((qid, quote))

print(f"נמצאו {len(coverage) - len(missing)} מתוך {len(coverage)} ציטוטים")
for qid, quote in missing:
    print(f"  חסר: {qid} — {quote}")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

# הקידומות שהמודל הזה צריך: query לשאלות, passage לקטעים
passage_texts = ["passage: " + s for s in segments]
query_texts = ["query: " + t for t in questions['text']]

passage_emb = model.encode(passage_texts, normalize_embeddings=True, show_progress_bar=True)
query_emb = model.encode(query_texts, normalize_embeddings=True)

print(passage_emb.shape, query_emb.shape)

In [ ]:
sims = model.similarity(query_emb, passage_emb)
print(sims.shape)

In [ ]:
qid = 'q12'
row = questions.index[questions['id'] == qid][0]
print(row, questions.loc[row, 'text'])

In [ ]:
scores, idx = sims[row].topk(5)

for score, i in zip(scores.tolist(), idx.tolist()):
    print(f"{score:.3f}  {segments.iloc[i]}")

In [ ]:
stance, confidence, quote = coverage[qid]
top_segments = [segments.iloc[i] for i in idx.tolist()]
hit = any(quote in s for s in top_segments)
print(qid, "נמצא" if hit else "לא נמצא")

In [ ]:
k = 5
hits = []
for qid in coverage:
    row = questions.index[questions['id'] == qid][0]
    scores, idx = sims[row].topk(k)
    stance, confidence, quote = coverage[qid]
    top_segments = [segments.iloc[i] for i in idx.tolist()]
    hit = any(quote in s for s in top_segments)
    hits.append(hit)
    if not hit:
        print(f"פספוס: {qid} — המודל בחר ראשון: {top_segments[0][:80]}")

recall = sum(hits) / len(hits)
print(f"recall@{k} = {recall:.2f}")

In [ ]:
def recall_at_k(k):
    hits = []
    for qid in coverage:
        row = questions.index[questions['id'] == qid][0]
        scores, idx = sims[row].topk(k)
        stance, confidence, quote = coverage[qid]
        top_segments = [segments.iloc[i] for i in idx.tolist()]
        hits.append(any(quote in s for s in top_segments))
    return sum(hits) / len(hits)


for k in [1, 3, 5, 10]:
    r = recall_at_k(k)
    print(f"recall@{k:<2} = {r:.2f}   ({round(r * len(coverage))} מתוך {len(coverage)})")

# שלב ב' – ניסוי 1: שליפת קטעים מהמצע לפי שאלה

## מה נבדק
האם מודל embeddings מוצא במצע את הקטעים שעונים על כל שאלה בשאלון.
הבדיקה נעשתה מול התיוג הידני של מצע ישראל ביתנו: 15 שאלות, ולכל אחת ציטוט אחד שנבחר ידנית.

## השיטה
- **מודל:** `intfloat/multilingual-e5-base`, עם הקידומות `query:` לשאלות ו-`passage:` לקטעים.
- **פיצול:** לפי שורות, אחרי הסרת שורות ריקות, כותרות, שורות של מילה אחת ושורות של עד 10 תווים. נשארו 563 קטעים, באורך חציוני של 68 תווים.
- **דמיון:** Cosine Similarity בין כל שאלה לכל קטע.
- **מדד:** recall@k, כלומר בכמה מהשאלות הציטוט שנבחר ידנית נמצא בין k הקטעים שהמודל דירג הכי גבוה.

## תוצאות

| k | recall | מתוך 15 |
|---|---|---|
| 1 | 0.07 | 1 |
| 3 | 0.20 | 3 |
| 5 | 0.40 | 6 |
| 10 | 0.53 | 8 |

## ניתוח הפספוסים (k=5)
מתוך 9 פספוסים:
- **4 פספוסים שאינם טעות של המודל** (`q04`, `q05`, `q14`, `q19`): המודל בחר קטע אחר שגם הוא עונה על השאלה. המדד מכיר רק תשובה נכונה אחת לכל שאלה, ולכן הוא מעניש את המודל גם כשהוא צודק.
- **5 טעויות אמיתיות** (`q07`, `q08`, `q10`, `q21`, `q24`): הקטע שנבחר לא קשור לשאלה.

אם סופרים גם את הקטעים הנכונים "האחרים", המודל מצא קטע רלוונטי ב-10 מתוך 15 שאלות (כ-0.67).

## תבניות שזוהו
1. **שאלות מופשטות:** הטעויות מרוכזות בכלכלה ובחברה. השאלות מנוסחות באופן כללי ("המדינה צריכה להתערב במחירים"), והמצע מדבר בפרטים ("רשת ביטחון לעצמאים", "פירוק מונופולים").
2. **קטעים ש"מושכים" הכל (hubness):** הקטע על רשת ביטחון לעצמאים יצא ראשון בשתי שאלות שאין ביניהן קשר (`q07`, `q24`).
3. **כפילויות:** המצע חוזר על עצמו (סיכום ופירוט), וקטעים כמעט זהים תופסים כמה מקומות ב-top-k.
4. **ציונים צפופים:** הפער בין המקום הראשון לחמישי הוא בערך 0.01. הסדר חשוב, לא הערך, ולכן אי אפשר לקבוע סף אחיד של "רלוונטי".
5. **דמיון של נושא כללי במקום נושא מדויק:** `q10` (קצבאות למשפחות ולקשישים) קיבלה קטע על ענישה בגין פגיעה בקטינים. שניהם "הגנה על חלשים", אבל נושאים שונים.

## מסקנות
- המודל מתאים לשיטה שבה אדם מאשר: הוא מצמצם 563 קטעים לכ-10 מועמדים לכל שאלה.
- מומלץ להציג כ-10 קטעים לכל שאלה ולא 5, כי ה-recall ממשיך לעלות בין 5 ל-10.

## הצעדים הבאים
1. **לתקן את המדד:** לסמן לכל שאלה את כל הקטעים הנכונים, ולא רק אחד.
2. **להסיר קטעים כפולים.**
3. **להשוות למודל `BAAI/bge-m3`.**

את כל השיפורים יש למדוד מול המדד המתוקן.

In [ ]:
def show_candidates(qid, k=10):
    row = questions.index[questions['id'] == qid][0]
    print(f"{qid}: {questions.loc[row, 'text']}")
    stance, confidence, quote = coverage[qid]
    print(f"הציטוט שבחרת: {quote}\n")
    scores, idx = sims[row].topk(k)
    for rank, (score, i) in enumerate(zip(scores.tolist(), idx.tolist()), start=1):
        mark = "  ★" if quote in segments.iloc[i] else ""
        print(f"{rank:>2}. [{i:>3}] {score:.3f}  {segments.iloc[i][:150]}{mark}")
    print()

show_candidates('q12')

In [ ]:
def show_candidates(qid, k=10):
    row = questions.index[questions['id'] == qid][0]
    stance, confidence, quote = coverage[qid]
    print(f"{qid}: {questions.loc[row, 'text']}")
    print(f"הציטוט שבחרת קודם: {quote}\n")
    scores, idx = sims[row].topk(k)
    for rank, (score, i) in enumerate(zip(scores.tolist(), idx.tolist()), start=1):
        mark = "  ★" if quote in segments.iloc[i] else ""
        print(f"{rank:>2}. [{i:>3}] {segments.iloc[i][:150]}{mark}")
    print()


def ask_gold(qid):
    show_candidates(qid)
    answer = input(f"{qid}: מספרי הקטעים הנכונים, מופרדים בפסיק (Enter אם אין): ")
    chosen = [int(x) for x in answer.replace(' ', '').split(',') if x]
    # הקטעים שמכילים את הציטוט שבחרת קודם נכנסים תמיד, גם אם לא הופיעו בעשירייה
    stance, confidence, quote = coverage[qid]
    from_quote = [i for i in range(len(segments)) if quote in segments.iloc[i]]
    gold[qid] = sorted(set(chosen) | set(from_quote))
    print(f"נשמר: {qid} → {gold[qid]}\n" + "-" * 60)


gold = {}
qids = list(coverage)
print(f"{len(qids)} שאלות לתיוג:", qids)

In [ ]:
for qid in qids[0:3]:
    ask_gold(qid)

In [ ]:
gold

In [ ]:
ask_gold('q04')

In [ ]:
ask_gold(qids[3])

In [ ]:
ask_gold(qids[4])

In [ ]:
ask_gold(qids[5])

In [ ]:
ask_gold(qids[6])

In [ ]:
ask_gold(qids[7])

In [ ]:
ask_gold(qids[8])

In [ ]:
ask_gold(qids[9])

In [ ]:
ask_gold(qids[10])

In [ ]:
ask_gold(qids[11])

In [ ]:
ask_gold(qids[12])

In [ ]:
ask_gold(qids[13])

In [ ]:
ask_gold(qids[14])

In [ ]:
gold

In [ ]:
def recall_at_k_gold(k):
    hits = []
    for qid, correct in gold.items():
        row = questions.index[questions['id'] == qid][0]
        scores, idx = sims[row].topk(k)
        hits.append(bool(set(idx.tolist()) & set(correct)))
    return sum(hits) / len(hits)


print("k    ישן    מתוקן")
for k in [1, 3, 5, 10]:
    print(f"{k:<4} {recall_at_k(k):.2f}   {recall_at_k_gold(k):.2f}")

In [ ]:
import json
json.dump(gold, open('gold_yisrael-beitenu.json', 'w'), indent=1)